### Convert dac to wav:

In [1]:
!pip install descript-audio-codec

In [2]:
from pathlib import Path
import dac

# Download a model
model_path = dac.utils.download(model_type="44khz")
model = dac.DAC.load(model_path)

# Assuming ⁠ model ⁠ is already defined somewhere in your script
input_dir = Path("testdata/dac-train")
output_dir = input_dir / "output"
output_dir.mkdir(exist_ok=True)  # Create output directory if it doesn't exist

for dac_file in input_dir.glob("*.dac"):
    print(f"Processing {dac_file}...")

    # Load the DAC file
    x = dac.DACFile.load(dac_file)

    # Decompress it back to an AudioSignal
    y = model.decompress(x)

    # Define output file path
    output_file = output_dir / f"{dac_file.stem}.wav"

    # Write to file
    y.write(output_file)

    print(f"Saved to {output_file}")

/Users/malu/Documents/upf/term 2/computational music creativity/DACSynthformer-main/.venv/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:28: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")


Processing testdata/dac-train/DSBugs--busybodyFreqFactor-00.50--c-00--x-90.dac...
Saved to testdata/dac-train/output/DSBugs--busybodyFreqFactor-00.50--c-00--x-90.wav
Processing testdata/dac-train/DSWind--strength-00.50--c-00--x-90.dac...
Saved to testdata/dac-train/output/DSWind--strength-00.50--c-00--x-90.wav
Processing testdata/dac-train/DSApplause--numClappers_exp-00.50--c-02--x-92.dac...
Saved to testdata/dac-train/output/DSApplause--numClappers_exp-00.50--c-02--x-92.wav
Processing testdata/dac-train/DSPistons--rate_exp-00.50--c-00--x-90.dac...
Saved to testdata/dac-train/output/DSPistons--rate_exp-00.50--c-00--x-90.wav


### Extract pitch

In [1]:
import essentia.standard as es
import numpy as np
import os

# Set the directory containing audio files
audio_dir = "testdata/dac-train/output"

# List all audio files (supports .wav and .mp3)
audio_files = [f for f in os.listdir(audio_dir) if f.endswith(('.wav', '.mp3'))]

def process_audio(file_path):
    """Extract and normalize pitch values for a given audio file."""
    audio = es.MonoLoader(filename=file_path)()
    pitch_extractor = es.PitchYinFFT()
    pitch_values = []

    for frame in es.FrameGenerator(audio, frameSize=1024, hopSize=512):
        pitch, _ = pitch_extractor(frame)
        pitch_values.append(pitch)

    pitch_values = np.array(pitch_values)
    pitch_values[pitch_values == 0] = np.nan  # Ignore unvoiced frames

    # Normalize between 0 and 1
    min_pitch, max_pitch = np.nanmin(pitch_values), np.nanmax(pitch_values)
    normalized_pitch = (pitch_values - min_pitch) / (max_pitch - min_pitch)
    normalized_pitch = np.nan_to_num(normalized_pitch)  # Replace NaNs with 0

    return normalized_pitch

# Loop through all files and process them
for file in audio_files:
    file_path = os.path.join(audio_dir, file)
    print(f"\nProcessing: {file}")

    # Extract and normalize pitch
    normalized_pitch = process_audio(file_path)

    # Print the first 10 normalized pitch values (to keep output manageable)
    print("First 10 normalized pitch values:", np.mean(normalized_pitch[:]))

print("\nProcessing complete!")


Processing: DSPistons--rate_exp-00.50--c-00--x-90.wav
First 10 normalized pitch values: 0.17948749829365265

Processing: DSApplause--numClappers_exp-00.50--c-02--x-92.wav
First 10 normalized pitch values: 0.16540227416051548

Processing: DSWind--strength-00.50--c-00--x-90.wav
First 10 normalized pitch values: 0.030126289511698347

Processing: DSBugs--busybodyFreqFactor-00.50--c-00--x-90.wav
First 10 normalized pitch values: 0.06404272009619831

Processing complete!


### Extract bpm

In [16]:
import essentia.standard as es
import numpy as np
import os

def extract_bpm(file_path):
    """Extract BPM from a .wav file using RhythmExtractor2013."""
    
    # Load audio file
    audio = es.MonoLoader(filename=file_path)()
    
    # Rhythm extractor using "multifeature" method
    rhythm_extractor = es.RhythmExtractor2013(method="multifeature")
    
    # Extract BPM and other rhythm features
    bpm, beats, beats_confidence, _, beats_intervals = rhythm_extractor(audio)
    
    return bpm

# Set the directory containing the audio files
audio_dir = "testdata/dac-train/output"

# List all audio files (supports .wav and .mp3)
audio_files = [f for f in os.listdir(audio_dir) if f.endswith(('.wav', '.mp3'))]

# Extract BPM for each file and store them
bpm_values = []

# Loop through all files and extract BPM
for file in audio_files:
    file_path = os.path.join(audio_dir, file)
    print(f"\nProcessing: {file}")
    
    # Extract BPM
    bpm = extract_bpm(file_path)
    bpm_values.append(bpm)

# Now, normalize the BPM values across all files
min_bpm, max_bpm = np.min(bpm_values), np.max(bpm_values)

# Normalize BPM values between 0 and 1
normalized_bpm_values = [(bpm - min_bpm) / (max_bpm - min_bpm) if max_bpm != min_bpm else bpm for bpm in bpm_values]

# Print normalized BPM for each file
for file, normalized_bpm in zip(audio_files, normalized_bpm_values):
    print(f"Normalized BPM for {file}: {normalized_bpm}")

print("\nProcessing complete!")


Processing: DSPistons--rate_exp-00.50--c-00--x-90.wav

Processing: DSApplause--numClappers_exp-00.50--c-02--x-92.wav

Processing: DSWind--strength-00.50--c-00--x-90.wav

Processing: DSBugs--busybodyFreqFactor-00.50--c-00--x-90.wav
Normalized BPM for DSPistons--rate_exp-00.50--c-00--x-90.wav: 0.4608414911008717
Normalized BPM for DSApplause--numClappers_exp-00.50--c-02--x-92.wav: 0.21148411649767934
Normalized BPM for DSWind--strength-00.50--c-00--x-90.wav: 0.0
Normalized BPM for DSBugs--busybodyFreqFactor-00.50--c-00--x-90.wav: 1.0

Processing complete!


### Extract Loudness

In [ ]:
import essentia.standard as es
import numpy as np
import os

def extract_and_normalize_loudness(file_path):
    """Extract loudness from a .wav file and normalize it between 0 and 1."""
    
    # Load audio file
    audio = es.MonoLoader(filename=file_path)()
    
    # Loudness extractor
    loudness_extractor = es.Loudness()
    loudness_values = []
    
    # Process the audio in frames
    for frame in es.FrameGenerator(audio, frameSize=1024, hopSize=512):
        loudness = loudness_extractor(frame)  
        loudness_values.append(loudness)
    
    # Convert loudness values to a numpy array
    loudness_values = np.array(loudness_values)

    # Normalize the loudness values between 0 and 1
    if len(loudness_values) > 0:
        min_loudness, max_loudness = np.min(loudness_values), np.max(loudness_values)
        normalized_loudness = (loudness_values - min_loudness) / (max_loudness - min_loudness) if max_loudness > min_loudness else loudness_values
    else:
        normalized_loudness = np.array([])  
    
    return normalized_loudness

# Load all the .wav files
file_path = "testdata/dac-train/output"

if os.path.isdir(file_path):
    wav_files = [f for f in os.listdir(file_path) if f.endswith(".wav")]
    if not wav_files:
        print("Error: No .wav files found in the directory.")
    else:
        for wav_file in wav_files:
            wav_path = os.path.join(file_path, wav_file)
            print(f"Processing file: {wav_path}")
            normalized_loudness = extract_and_normalize_loudness(wav_path)
            print(f"First 10 normalized loudness values for {wav_file}: {np.mean(normalized_loudness[:10])}")
else:
    if not os.path.exists(file_path):
        print("Error: The file does not exist.")
    elif not file_path.endswith(".wav"):
        print("Error: The provided file is not a .wav file.")
    else:
        normalized_loudness = extract_and_normalize_loudness(file_path)
        print("First 10 normalized loudness values:", np.mean(normalized_loudness))


Processing file: testdata/dac-train/output/DSPistons--rate_exp-00.50--c-00--x-90.wav
First 10 normalized loudness values for DSPistons--rate_exp-00.50--c-00--x-90.wav: 0.22689303755593437
Processing file: testdata/dac-train/output/DSApplause--numClappers_exp-00.50--c-02--x-92.wav
First 10 normalized loudness values for DSApplause--numClappers_exp-00.50--c-02--x-92.wav: 0.10852369048392645
Processing file: testdata/dac-train/output/DSWind--strength-00.50--c-00--x-90.wav
First 10 normalized loudness values for DSWind--strength-00.50--c-00--x-90.wav: 0.1939560481809876
Processing file: testdata/dac-train/output/DSBugs--busybodyFreqFactor-00.50--c-00--x-90.wav
First 10 normalized loudness values for DSBugs--busybodyFreqFactor-00.50--c-00--x-90.wav: 0.15561072984590948
